# VarianceThreshold

Drops every feature whose variance is below a threshold.

It never looks at the target, so it is unsupervised. Its job is cleanup, not
ranking: a column that barely changes cannot tell samples apart.

As in scikit-learn, `transform` returns a NumPy array. Here
`set_output(transform='pandas')` keeps the column names.

- **Cost:** one pass over the data. The fastest selector here.
- **Strength:** removes constant and near-constant columns before any other step.
- **Weakness:** variance depends on units. After standardization every column
  has variance 1, and the selector keeps everything.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
from featsel.selectors import VarianceThreshold

X = pd.DataFrame({
    'varies':   rng.normal(0, 1.0, 200),
    'constant': np.full(200, 3.0),
    'tiny':     rng.normal(0, 0.01, 200),
    'wide':     rng.normal(0, 10.0, 200),
})
X.var().round(4)

varies        0.9285
constant      0.0000
tiny          0.0001
wide        100.4327
dtype: float64

## Dropping constant and near-constant columns

In [2]:
for threshold in (0.0, 0.001, 1.0):
    selector = VarianceThreshold(threshold=threshold).set_output(transform='pandas')
    kept = selector.fit(X).transform(X).columns.tolist()
    print(f'threshold {threshold:<6} keeps {kept}')

threshold 0.0    keeps ['varies', 'tiny', 'wide']
threshold 0.001  keeps ['varies', 'wide']
threshold 1.0    keeps ['wide']


## The unit trap

After standardization every varying column has variance 1. The constant
column is still dropped, but `tiny` can no longer be told apart from `wide`.
Run it before scaling, not after.

In [3]:
X_scaled = (X - X.mean()) / X.std().replace(0, 1)
selector = VarianceThreshold(threshold=0.5).set_output(transform='pandas')
selector.fit(X_scaled).transform(X_scaled).columns.tolist()

['varies', 'tiny', 'wide']